In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv("voxel_map.txt", comment="#")

def write_vtu(df, filename):
    n = len(df)

    points = "\n".join(
        f"{row.x_mm} {row.y_mm} {row.z_mm}"
        for row in df.itertuples()
    )

    pe_data = "\n".join(str(v) for v in df["energyPE_eV"])
    ke_data = "\n".join(str(v) for v in df["energyKE_eV"])

    vtu = f"""<?xml version="1.0"?>
<VTKFile type="UnstructuredGrid" version="0.1" byte_order="LittleEndian">
  <UnstructuredGrid>
    <Piece NumberOfPoints="{n}" NumberOfCells="{n}">

      <PointData>
        <DataArray type="Float64" Name="energyPE_eV" format="ascii">
          {pe_data}
        </DataArray>
        <DataArray type="Float64" Name="energyKE_eV" format="ascii">
          {ke_data}
        </DataArray>
      </PointData>

      <Points>
        <DataArray type="Float64" NumberOfComponents="3" format="ascii">
          {points}
        </DataArray>
      </Points>

      <Cells>
        <DataArray type="Int32" Name="connectivity" format="ascii">
          {" ".join(str(i) for i in range(n))}
        </DataArray>
        <DataArray type="Int32" Name="offsets" format="ascii">
          {" ".join(str(i+1) for i in range(n))}
        </DataArray>
        <DataArray type="UInt8" Name="types" format="ascii">
          {"1 " * n}
        </DataArray>
      </Cells>

    </Piece>
  </UnstructuredGrid>
</VTKFile>"""

    with open(filename, "w") as f:
        f.write(vtu)

write_vtu(df, "voxel_map.vtu")
print(f"Written {len(df)} voxels to voxel_map.vtu")

Written 51163 voxels to voxel_map.vtu


In [11]:
df["energyPE_eV"].sum() + df["energyKE_eV"].sum()

np.float64(99587746.258733)

In [12]:
import uproot 

with uproot.open("output.root") as f:
    print(f.keys())          


['particles;1', 'photons;1', 'escaped;1']


In [13]:
with uproot.open("output.root") as f:
    tree = f["photons"]
    df_photons = tree.arrays(library="pd")
    tree= f["escaped"]
    df_escaped = tree.arrays(library="pd")




In [14]:
df_photons['energy'].sum()

np.float64(412146.07)

In [15]:
df["energyPE_eV"].sum() + df["energyKE_eV"].sum() + df_photons['energy'].sum() + df_escaped['energy'].sum()

np.float64(100000000.00011726)